# Train model

In [1]:
import sys
from pathlib import Path

# Kaggle:
sys.path.insert(0, "/kaggle/input/datasets/gpla77/pro5-code")
DATA_PATH = Path("/kaggle/input/datasets/gpla77/pro5-data/train.npz")
CKPT_DIR  = Path("/kaggle/working/checkpoints")
NORM_STATS_PATH = Path("/kaggle/input/datasets/gpla77/pro5-data/norm_stats.npy")

# Local:
# DATA_PATH = Path("data/train.npz")
# CKPT_DIR  = Path("checkpoints")
# NORM_STATS_PATH = Path(data/norm_stats.npz")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Data   : {DATA_PATH}  (exists: {DATA_PATH.exists()})")
print(f"Norm stats: {NORM_STATS_PATH} (exists: {NORM_STATS_PATH.exists()})")

Device : cuda
GPU    : Tesla T4
Data   : /kaggle/input/datasets/gpla77/pro5-data/train.npz  (exists: True)
Norm stats: /kaggle/input/datasets/gpla77/pro5-data/norm_stats.npy (exists: True)


# Dataset

In [2]:
from dataset import MotionDataset

dataset = MotionDataset(str(DATA_PATH))
seq, label = dataset[0]
print(f"Samples        : {len(dataset)}")
print(f"Sequence shape : {seq.shape}")   # [T, J, 3]
print(f"Num classes    : {int(dataset.labels.max()) + 1}")

Samples        : 1260
Sequence shape : torch.Size([48, 15, 3])
Num classes    : 2


# Train

In [3]:
from train import train

model = train(
    dataset        = dataset,
    norm_stats_path= str(NORM_STATS_PATH),
    # model
    d_model        = 384,
    nhead          = 6,
    num_layers     = 6,
    dropout        = 0.1,
    # diffusion
    timesteps      = 1000,
    beta_start     = 1e-4,
    beta_end       = 0.02,
    cfg_drop_prob  = 0.1,
    guidance_scale = 3.0,
    # training
    epochs         = 3600,
    batch_size     = 64,
    lr             = 2e-10,
    optimizer      = "adam",
    weight_decay   = 1e-4,
    scheduler      = "cosine",
    grad_clip      = 1.0,
    vel_loss_weight= 2,
    save_every     = 50,
    # qualitative eval
    eval_every     = 50,
    eval_samples   = 4,
    # misc
    device         = DEVICE,
    ckpt_dir       = str(CKPT_DIR),
    # Kaggle resume:
    resume_from  = "/kaggle/input/datasets/gpla77/pro5-model/ckpt_e3850.pt",
    resume_optimizer=False,
    resume_scheduler=False, 
    seed           = 42,
    num_workers    = 0,   
)

MotionDenoiser  params: 11,292,333  |  device: cuda
dataset: 1260  classes: 2
resumed from: /kaggle/input/datasets/gpla77/pro5-model/ckpt_e3850.pt  (epoch 3851)

Done → /kaggle/working/checkpoints/final_model.pt


In [4]:
from pathlib import Path
from sample import visualize_training_samples

for pt_file in sorted(Path("checkpoints/samples").glob("samples_e*.pt")):
    visualize_training_samples(
        samples_pt = str(pt_file),
        save_dir   = f"checkpoints/samples/gifs/{pt_file.stem}",
        fps        = 4,
    )